# Day 1 — SQL: Filters (WHERE, HAVING, BETWEEN, IN, LIKE)
### Target: Data Engineer Interviews | PostgreSQL

> **Roadmap Day:** 1 · **Date:** Monday, June 15, 2026  
> **Dataset:** `employees` table from `setup_tables.sql`  
> **Prerequisite:** Run `00_prerequisites.ipynb` first — PostgreSQL must be running and tables loaded.

---
## Setup — Connect to PostgreSQL

Change `PASSWORD` to match your PostgreSQL password.

In [ ]:
%load_ext sql

USERNAME = "postgres"
PASSWORD = "postgres"    # <-- change if yours is different
HOST     = "localhost"
PORT     = "5432"
DBNAME   = "postgres"

connection_string = f"postgresql://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}"
%sql $connection_string
print("Connected to PostgreSQL.")

In [ ]:
%%sql
-- Confirm employees table is loaded
SELECT COUNT(*) AS total_employees FROM employees;

---
## 1. The employees Table

All Day 1 queries use this table.  
Schema: `employees(id, name, email, department, department_id, job_title, salary, bonus, manager_id, hire_date, status, gender, age, phone)`

In [ ]:
%%sql
SELECT id, name, department, salary, bonus, hire_date, status
FROM employees
ORDER BY id;

---
## 2. WHERE — Row-Level Filtering

`WHERE` runs **before** any grouping. Filters individual rows.

In [ ]:
%%sql
-- Basic equality
SELECT id, name, department, salary
FROM employees
WHERE department = 'Engineering'
ORDER BY salary DESC;

In [ ]:
%%sql
-- Not equal — active employees only
SELECT id, name, department, status
FROM employees
WHERE status != 'inactive'
ORDER BY department;

In [ ]:
%%sql
-- Numeric comparison — high earners
SELECT id, name, department, salary
FROM employees
WHERE salary > 90000
ORDER BY salary DESC;

---
## 3. BETWEEN — Inclusive Range Filter

`BETWEEN low AND high` — **inclusive on both ends.**  
Equivalent to `>= low AND <= high`.

In [ ]:
%%sql
-- Salary range — inclusive on both ends
SELECT id, name, department, salary
FROM employees
WHERE salary BETWEEN 60000 AND 100000
ORDER BY salary;

In [ ]:
%%sql
-- NOT BETWEEN — outside the range
SELECT id, name, salary
FROM employees
WHERE salary NOT BETWEEN 60000 AND 100000
ORDER BY salary DESC;

In [ ]:
%%sql
-- BETWEEN on dates — hired in 2022
SELECT id, name, department, hire_date
FROM employees
WHERE hire_date BETWEEN '2022-01-01' AND '2022-12-31'
ORDER BY hire_date;

In [ ]:
%%sql
-- Alternative: DATE_TRUNC / EXTRACT for year filter — more robust for timestamps
SELECT id, name, hire_date
FROM employees
WHERE hire_date >= '2022-01-01'
  AND hire_date <  '2023-01-01'   -- safer than BETWEEN for timestamps
ORDER BY hire_date;

---
## 4. IN and NOT IN — Match Any Value in a List

In [ ]:
%%sql
-- IN — same as writing multiple OR conditions
SELECT id, name, department
FROM employees
WHERE department IN ('Engineering', 'Finance')
ORDER BY department, name;

In [ ]:
%%sql
-- NOT IN — exclude specific departments
SELECT id, name, department
FROM employees
WHERE department NOT IN ('HR', 'Marketing')
ORDER BY department, name;

In [ ]:
%%sql
-- IN vs OR — equivalent results
SELECT id, name, department
FROM employees
WHERE department = 'Engineering' OR department = 'Finance'
ORDER BY department, name;

In [ ]:
%%sql
-- IN with subquery — employees in departments with budget > 2M
SELECT e.id, e.name, e.department, e.salary
FROM employees e
WHERE e.department_id IN (
    SELECT id FROM departments WHERE budget > 2000000
)
ORDER BY e.department, e.salary DESC;

---
## 5. LIKE and ILIKE — Pattern Matching

| Wildcard | Meaning |
|----------|---------|
| `%` | Zero or more characters |
| `_` | Exactly one character |

PostgreSQL: `LIKE` is case-sensitive · `ILIKE` is case-insensitive

In [ ]:
%%sql
-- Names starting with 'A' — case sensitive
SELECT id, name, department
FROM employees
WHERE name LIKE 'A%';

In [ ]:
%%sql
-- ILIKE — case insensitive (PostgreSQL only)
SELECT id, name FROM employees WHERE name ILIKE 'a%';

In [ ]:
%%sql
-- Names ending with 'son'
SELECT id, name FROM employees WHERE name LIKE '%son';

In [ ]:
%%sql
-- Email at gmail
SELECT id, name, email FROM employees WHERE email LIKE '%@gmail.com';

In [ ]:
%%sql
-- _ wildcard — exactly one character
-- Names with exactly 8 characters
SELECT id, name, LENGTH(name) AS name_len
FROM employees
WHERE name LIKE '________'   -- eight underscores
ORDER BY name;

In [ ]:
%%sql
-- NOT LIKE
SELECT id, name, email
FROM employees
WHERE email NOT LIKE '%@test.%'
ORDER BY name;

---
## 6. AND / OR / NOT — Combining Conditions

**Operator precedence:** `AND` evaluates before `OR` — always use parentheses when mixing.

In [ ]:
%%sql
-- AND — both conditions must be true
SELECT id, name, department, salary
FROM employees
WHERE department = 'Engineering'
  AND salary BETWEEN 60000 AND 100000
ORDER BY salary DESC;

In [ ]:
%%sql
-- OR — either condition is sufficient
-- Name starts with 'A' OR hired in 2022
SELECT id, name, department, hire_date
FROM employees
WHERE name LIKE 'A%'
   OR hire_date BETWEEN '2022-01-01' AND '2022-12-31'
ORDER BY hire_date;

In [ ]:
%%sql
-- NOT
SELECT id, name, department
FROM employees
WHERE NOT department = 'HR'
ORDER BY department, name;

In [ ]:
%%sql
-- Precedence gotcha: AND evaluates before OR
-- These two queries return DIFFERENT results

-- Query A: no parentheses — AND binds first, so reads as: name LIKE 'A%' OR (dept='HR' AND salary>50000)
SELECT id, name, department, salary
FROM employees
WHERE name LIKE 'A%' OR department = 'HR' AND salary > 50000;

In [ ]:
%%sql
-- Query B: explicit parentheses — (name LIKE 'A%' OR dept='HR') AND salary>50000
SELECT id, name, department, salary
FROM employees
WHERE (name LIKE 'A%' OR department = 'HR') AND salary > 50000;

---
## 7. NULL Handling

NULL is the **absence of a value**. Always use `IS NULL` / `IS NOT NULL` — never `= NULL`.

In [ ]:
%%sql
-- Employees with no manager (top-level managers)
SELECT id, name, department, manager_id
FROM employees
WHERE manager_id IS NULL
ORDER BY department;

In [ ]:
%%sql
-- Employees with no phone number on file
SELECT id, name, phone
FROM employees
WHERE phone IS NULL;

In [ ]:
%%sql
-- = NULL never returns rows — classic bug
SELECT COUNT(*) AS wrong_count  FROM employees WHERE phone = NULL;       -- returns 0
-- vs
-- SELECT COUNT(*) AS right_count FROM employees WHERE phone IS NULL;

In [ ]:
%%sql
-- COALESCE — return first non-null value
SELECT id, name,
       COALESCE(phone, email, 'no contact') AS contact_info
FROM employees
ORDER BY id;

---
## 8. HAVING — Filter After Grouping

`WHERE` cannot use aggregate functions. `HAVING` runs after `GROUP BY` and can.

**SQL Execution Order:**
```
1. FROM      → identify source
2. WHERE     → filter rows         ← no aggregates here
3. GROUP BY  → form groups
4. HAVING    → filter groups       ← aggregates OK here
5. SELECT    → compute output
6. ORDER BY  → sort
7. LIMIT     → paginate
```

In [ ]:
%%sql
-- Departments where average salary > 70000
SELECT department,
       ROUND(AVG(salary), 2) AS avg_salary,
       COUNT(*)               AS headcount
FROM employees
GROUP BY department
HAVING AVG(salary) > 70000
ORDER BY avg_salary DESC;

In [ ]:
%%sql
-- All departments — for comparison
SELECT department,
       ROUND(AVG(salary), 0) AS avg_salary,
       COUNT(*)               AS headcount,
       MIN(salary)            AS min_sal,
       MAX(salary)            AS max_sal
FROM employees
GROUP BY department
ORDER BY avg_salary DESC;

In [ ]:
%%sql
-- WHERE + HAVING together
-- Active employees only (WHERE) → depts with 2+ headcount and avg > 60000 (HAVING)
SELECT department,
       COUNT(*)               AS headcount,
       ROUND(AVG(salary), 0)  AS avg_salary
FROM employees
WHERE status = 'active'        -- row-level filter
GROUP BY department
HAVING COUNT(*) >= 2           -- group-level filter
   AND AVG(salary) > 60000
ORDER BY avg_salary DESC;

In [ ]:
%%sql
-- WHERE with aggregate — ERROR (expected)
-- This demonstrates why HAVING exists
SELECT department, AVG(salary)
FROM employees
WHERE AVG(salary) > 70000     -- ERROR: aggregate functions not allowed in WHERE
GROUP BY department;

---
## 9. Day 1 Problems — Official Solutions

These are the exact problems from the roadmap.  
Try writing your solution before expanding.

In [ ]:
%%sql
-- Problem 1 (Easy)
-- Find all employees in the 'Engineering' department with salary between 60000 and 100000

SELECT id, name, department, salary
FROM employees
WHERE department = 'Engineering'
  AND salary BETWEEN 60000 AND 100000
ORDER BY salary DESC;

In [ ]:
%%sql
-- Problem 2 (Easy)
-- Find departments where the average salary is greater than 70000

SELECT department,
       ROUND(AVG(salary), 2) AS avg_salary,
       COUNT(*)               AS headcount
FROM employees
GROUP BY department
HAVING AVG(salary) > 70000
ORDER BY avg_salary DESC;

In [ ]:
%%sql
-- Problem 3 (Medium)
-- Find employees whose name starts with 'A' OR whose hire_date falls in the year 2022

SELECT id, name, department, salary, hire_date
FROM employees
WHERE name LIKE 'A%'
   OR hire_date BETWEEN '2022-01-01' AND '2022-12-31'
ORDER BY hire_date;

In [ ]:
%%sql
-- Problem 3 — alternative using EXTRACT (PostgreSQL)
SELECT id, name, department, hire_date,
       CASE WHEN name LIKE 'A%' THEN 'name starts with A' ELSE '' END ||  
       CASE WHEN EXTRACT(YEAR FROM hire_date) = 2022 THEN 'hired in 2022' ELSE '' END AS matched_by
FROM employees
WHERE name LIKE 'A%'
   OR EXTRACT(YEAR FROM hire_date) = 2022
ORDER BY hire_date;

---
## 10. Interview Patterns — Must-Know Queries

In [ ]:
%%sql
-- Q1: Employees earning above the company average salary
SELECT id, name, department, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees)
ORDER BY salary DESC;

In [ ]:
%%sql
-- Q2: Top-level managers (no manager above them)
SELECT id, name, department, job_title
FROM employees
WHERE manager_id IS NULL
ORDER BY department;

In [ ]:
%%sql
-- Q3: Active senior engineers or managers — combine multiple conditions
SELECT id, name, department, job_title, salary
FROM employees
WHERE status = 'active'
  AND (
      job_title LIKE '%Senior%'
      OR job_title LIKE '%Manager%'
  )
ORDER BY salary DESC;

In [ ]:
%%sql
-- Q4: Departments with more than 2 active employees AND avg salary > 65000
SELECT department,
       COUNT(*)              AS active_count,
       ROUND(AVG(salary), 0) AS avg_salary,
       SUM(salary)           AS total_payroll
FROM employees
WHERE status = 'active'
GROUP BY department
HAVING COUNT(*) > 2
   AND AVG(salary) > 65000
ORDER BY avg_salary DESC;

In [ ]:
%%sql
-- Q5: Employees hired in the last 3 years with salary between 50000 and 80000
SELECT id, name, department, salary, hire_date
FROM employees
WHERE hire_date >= CURRENT_DATE - INTERVAL '3 years'
  AND salary BETWEEN 50000 AND 80000
ORDER BY hire_date DESC;

---
## Quick Cheat Sheet

```sql
-- Row-level filter
WHERE col = 'value'
WHERE col != 'value'
WHERE col BETWEEN 60000 AND 100000     -- inclusive
WHERE col IN ('a', 'b', 'c')
WHERE col NOT IN ('a', 'b')
WHERE col LIKE 'A%'                    -- starts with A
WHERE col ILIKE 'a%'                   -- case insensitive (PostgreSQL)
WHERE col IS NULL
WHERE col IS NOT NULL

-- Combining
WHERE a AND b                          -- both must be true
WHERE a OR b                           -- either is enough
WHERE (a OR b) AND c                   -- always parenthesize mixed AND/OR

-- Group-level filter
GROUP BY dept
HAVING AVG(salary) > 70000             -- runs after GROUP BY
HAVING COUNT(*) >= 2

-- Execution order
-- FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```